# 02 · Restore a saved run

Start of every later session: brings back a saved run's model, checkpoints, reports and data splits, and rebuilds the image caches (no `prepare_data` needed).

**Kaggle settings:** Accelerator *GPU T4 x2* (the code uses one GPU), Internet *on*,
secret `HF_TOKEN` (a Hugging Face token with write access) switched on for this notebook.
**Inputs:** APTOS 2019 Blindness Detection (competition); IDRiD (`aaryapatel98/indian-diabetic-retinopathy-image-dataset`).

`%%bash` must be the first line of its cell; put notes in markdown cells, not above it.

In [ ]:
import os, json
os.chdir('/kaggle/working')
if not os.path.exists('retina-screen'):
    !git clone -q https://github.com/ashwinruke/retina-screen.git
%cd /kaggle/working/retina-screen
!git pull -q
!pip install -q timm albumentations onnx onnxscript onnxruntime
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
!nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
TAG = 'a1'   # mvp (A0), a1, a2, a3 or a4
!python scripts/artifacts.py list
!python scripts/artifacts.py pull --tag {TAG}

In [ ]:
%%bash
set -e
cd /kaggle/working/retina-screen
python scripts/cache_images.py --manifest data/manifests/aptos.csv --out-dir cache --config configs/finetune_aptos.yaml
python scripts/cache_images.py --manifest data/manifests/idrid.csv --out-dir cache --config configs/finetune_aptos.yaml

## Check the restored model
Re-scores APTOS test with the restored model and its frozen cut points; should match the saved report to ~3 decimals.

In [ ]:
report = {'mvp': 'mvp_aptos_test', 'a1': 'A1_aptos_test', 'a3': 'A3_aptos_test'}.get(TAG)
if report:
    !python scripts/evaluate.py --checkpoint outputs/model.pt --manifest cache/aptos.csv --splits test --name check --out-dir outputs/check
    saved = json.load(open(f'outputs/eval/{report}.json'))['tuned_thresholds']['qwk']
    now = json.load(open('outputs/check/check.json'))['tuned_thresholds']['qwk']
    print(f"saved QWK {saved:.4f} | restored model QWK {now:.4f}")